In [163]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_absolute_error
from statsmodels.tsa.stattools import adfuller
import re
import ast
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from itertools import product
import torch.utils.data as data
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV,  GridSearchCV
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor

In [117]:
tart=pd.read_csv(r'C:\Users\zions\Cocci\ValleyFever-MediaSignals\data\raw\articlewise_nlp_features.csv')

In [118]:
tart.columns

Index(['year_month', 'query_term', 'title', 'published', 'link', 'source',
       'summary', 'real_url', 'article_text', 'article_detail',
       'source classes', 'County', 'title_missing', 'body_missing',
       'title_label', 'title_positive', 'title_neutral', 'title_negative',
       'title_sentiment', 'title_score', 'body_label', 'body_positive',
       'body_neutral', 'body_negative', 'body_sentiment', 'title_risk',
       'body_risk', 'date', 'month'],
      dtype='str')

In [119]:
tart=  (tart
    .groupby('year_month')
    .agg(
        article_count=('title', 'count'),
        mean_title_risk=('title_risk', 'mean'),
        mean_body_risk=('body_risk', 'mean'),
        mean_title_sentiment=('title_sentiment', 'mean'),
        mean_body_sentiment=('body_sentiment', 'mean')
    )
    .reset_index()
)

In [120]:
tart['year_month'] = (
    pd.to_datetime(tart['year_month'])
    .dt.to_period('M')
)

In [121]:
tart.columns

Index(['year_month', 'article_count', 'mean_title_risk', 'mean_body_risk',
       'mean_title_sentiment', 'mean_body_sentiment'],
      dtype='str')

In [122]:
Counties=pd.read_csv(r'C:\Users\zions\Cocci\ValleyFever-MediaSignals\data\raw\vf_clean_normal.csv')

In [123]:
Counties.head()

,County,Year-Month,num_cases,population,rate_per_100k
0,Alameda,2001-01,0.0,1469484.0,0.000000
1,Alameda,2001-02,0.0,1469484.0,0.000000
2,Alameda,2001-03,2.0,1469484.0,0.136102
3,Alameda,2001-04,2.0,1469484.0,0.136102
4,Alameda,2001-05,0.0,1469484.0,0.000000


In [124]:
Counties['year_month']=(
    pd.to_datetime(Counties['Year-Month'])
    .dt.to_period('M')
)

In [125]:
Counties= Counties[['County','year_month','num_cases','rate_per_100k']]

In [126]:
Counties = Counties.sort_values(['County', 'year_month'])

In [127]:
County_2001= Counties.merge(tart, on='year_month', how='inner')

In [128]:
County_2001

,County,year_month,num_cases,rate_per_100k,article_count,mean_title_risk,mean_body_risk,mean_title_sentiment,mean_body_sentiment
0,Alameda,2001-12,0.0,0.000000,1,0.000000,0.031519,-0.093663,-0.620226
1,Alameda,2004-05,0.0,0.000000,1,0.000000,0.007421,0.501221,0.591084
2,Alameda,2007-09,1.0,0.068695,1,0.000000,0.003717,0.011945,-0.007069
3,Alameda,2007-12,0.0,0.000000,1,0.090909,0.044037,-0.103591,-0.524382
4,Alameda,2009-02,4.0,0.266927,1,0.083333,0.067118,-0.078763,-0.195988
...,...,...,...,...,...,...,...,...,...
3439,Ventura,2024-08,22.0,2.655587,27,0.108151,0.048071,-0.288895,-0.312775
3440,Ventura,2024-09,22.0,2.655587,30,0.103014,0.043338,-0.188976,-0.243705
3441,Ventura,2024-10,30.0,3.621255,5,0.010000,0.013117,-0.047762,-0.218098
3442,Ventura,2024-11,45.0,5.431883,7,0.116625,0.027478,-0.053856,-0.076220


In [141]:
County_2001.isna().sum()

County                  0
year_month              0
num_cases               2
rate_per_100k           2
article_count           0
mean_title_risk         0
mean_body_risk          0
mean_title_sentiment    0
mean_body_sentiment     0
dtype: int64

In [145]:
County_2001.loc[County_2001['num_cases'].isna()]

,County,year_month,num_cases,rate_per_100k,article_count,mean_title_risk,mean_body_risk,mean_title_sentiment,mean_body_sentiment
820,Merced,2001-12,NaN,NaN,1,0.000000,0.031519,-0.093663,-0.620226
823,Merced,2007-12,NaN,NaN,1,0.090909,0.044037,-0.103591,-0.524382


In [146]:
County_2001.loc[County_2001['rate_per_100k'].isna()]

,County,year_month,num_cases,rate_per_100k,article_count,mean_title_risk,mean_body_risk,mean_title_sentiment,mean_body_sentiment
820,Merced,2001-12,NaN,NaN,1,0.000000,0.031519,-0.093663,-0.620226
823,Merced,2007-12,NaN,NaN,1,0.090909,0.044037,-0.103591,-0.524382


In [147]:
Counties.loc[Counties['year_month']=='2001-12']

,County,year_month,num_cases,rate_per_100k
11,Alameda,2001-12,0.0,0.000000
299,Contra Costa,2001-12,2.0,0.205983
587,Fresno,2001-12,2.0,0.245996
875,Kern,2001-12,88.0,13.006707
1163,Los Angeles,2001-12,7.0,0.072719
1451,Merced,2001-12,NaN,NaN
1739,Monterey,2001-12,0.0,0.000000
2027,Orange,2001-12,3.0,0.103970
2315,Riverside,2001-12,4.0,0.247428
2603,Sacramento,2001-12,1.0,0.078973


****Won't use Twitter data since there is a gap between 2001 and 2011**

In [134]:
adcols=['mean_title_sentiment', 'mean_title_risk', 'mean_body_sentiment',
       'mean_body_risk', 'article_count']

acols = {
    'articles': adcols
}

target='rate_per_100k'

horizons= [1,3,6]

In [135]:
for col in adcols:
    past_mean = County_2001[col].shift(1).expanding(min_periods=1).mean()
    County_2001[col] = County_2001[col].fillna(past_mean)

In [136]:
county_data = {
    county: group.sort_values('year_month').reset_index(drop=True)
    for county, group in County_2001.groupby('County')
}

In [137]:
county_model_data = {}

for county, df in county_data.items():
    df = df.sort_values('year_month').copy()

    df['log_rate'] = np.log1p(df[target])
    df['logdiff_rate'] = df['log_rate'].diff()

    df = df.dropna(subset=['logdiff_rate']).reset_index(drop=True)

    county_model_data[county] = {
        'df': df,
        'y': df['logdiff_rate'],
        'dates': df['year_month'],
        'X': {
    name: df[cols]
    for name, cols in acols.items()
}
}

In [138]:
for county, data in county_model_data.items():

    df = data['df']
    original = county_data[county].set_index('year_month')

    for col in acols['articles']:
        missing = df[col].isna()

        df.loc[missing, col] = df.loc[missing, 'year_month'].map(
            original[col]
        )

    data['X'] = {
        name: df[cols]
        for name, cols in acols.items()
    }

In [139]:
def walk_forward_splits(X, y, initial_train=120, horizon=1, step=1):
    splits = []

    for train_end in range(
        initial_train, len(X) - horizon + 1, step
    ):
        test_end = train_end + horizon

        splits.append({
            "X_train": X.iloc[:train_end],
            "X_test": X.iloc[train_end:test_end],
            "y_train": y.iloc[:train_end],
            "y_test": y.iloc[train_end:test_end]
        })

    return splits

initial_train = 120

county_splits = {}

for county, data in county_model_data.items():
    county_splits[county] = {}

    y = data['y']

    for feature_name, X in data['X'].items():
        county_splits[county][feature_name] = {}

        for horizon in horizons:
            county_splits[county][feature_name][horizon] = (
                walk_forward_splits(
                    X,
                    y,
                    initial_train=initial_train,
                    horizon=horizon,
                    step=1
                )
            )

In [92]:
param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [2, 3, 4, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [1, 5, 10]
}

In [93]:
XGB_results = {}
XGB_best_params = {}
XGB_importance_rows = []

for county, feature_sets_for_county in county_splits.items():

    for feature_name, horizons_for_feature in feature_sets_for_county.items():

        # Tune using the first training window
        initial_splits = horizons_for_feature[1]

        if not initial_splits:
            print(f"Skipping {county} / {feature_name}: insufficient data")
            continue

        first_split = initial_splits[0]

        X_initial = first_split["X_train"]
        y_initial = first_split["y_train"].to_numpy().ravel()

        # Hyperparameter search using chronological CV
        search = RandomizedSearchCV(
            estimator=XGBRegressor(
                objective="reg:squarederror",
                eval_metric="rmse",
                tree_method="hist",
                random_state=42,
                n_jobs=1
            ),
            param_distributions=param_distributions,
            n_iter=12,
            scoring="neg_mean_squared_error",
            cv=TimeSeriesSplit(n_splits=5),
            random_state=42,
            n_jobs=-1,
            verbose=0
        )

        search.fit(X_initial, y_initial)

        best_params = search.best_params_
        XGB_best_params[(county, feature_name)] = best_params

        print(f"Tuned {county} | {feature_name}")

        # Evaluate every horizon
        for horizon, splits_for_horizon in horizons_for_feature.items():

            forecasts = []

            for i, split in enumerate(splits_for_horizon):

                X_train = split["X_train"]
                X_test = split["X_test"]
                y_train = split["y_train"].to_numpy().ravel()
                y_test = split["y_test"].to_numpy().ravel()

                model = XGBRegressor(
                    **best_params,
                    objective="reg:squarederror",
                    eval_metric="rmse",
                    tree_method="hist",
                    random_state=42,
                    n_jobs=1
                )

                model.fit(X_train, y_train)
                predictions = model.predict(X_test)

                # Store predictions
                for lead, (actual, predicted) in enumerate(
                    zip(y_test, predictions), start=1
                ):
                    forecasts.append({
                        "split": i,
                        "lead": lead,
                        "actual": actual,
                        "predicted": predicted
                    })

                # Store feature importance for this fitted split
                for feature, importance in zip(
                    X_train.columns,
                    model.feature_importances_
                ):
                    XGB_importance_rows.append({
                        "County": county,
                        "Feature_Set": feature_name,
                        "Horizon": horizon,
                        "Split": i,
                        "Feature": feature,
                        "Importance": importance
                    })

            XGB_results[(county, feature_name, horizon)] = (
                pd.DataFrame(forecasts)
            )

            print(
                f"{county} | {feature_name} | horizon={horizon} | "
                f"{len(splits_for_horizon)} splits complete"
            )


Tuned Alameda | articles
Alameda | articles | horizon=1 | 43 splits complete
Alameda | articles | horizon=3 | 41 splits complete
Alameda | articles | horizon=6 | 38 splits complete
Tuned Contra Costa | articles
Contra Costa | articles | horizon=1 | 43 splits complete
Contra Costa | articles | horizon=3 | 41 splits complete
Contra Costa | articles | horizon=6 | 38 splits complete
Tuned Fresno | articles
Fresno | articles | horizon=1 | 43 splits complete
Fresno | articles | horizon=3 | 41 splits complete
Fresno | articles | horizon=6 | 38 splits complete
Tuned Kern | articles
Kern | articles | horizon=1 | 43 splits complete
Kern | articles | horizon=3 | 41 splits complete
Kern | articles | horizon=6 | 38 splits complete
Tuned Los Angeles | articles
Los Angeles | articles | horizon=1 | 43 splits complete
Los Angeles | articles | horizon=3 | 41 splits complete
Los Angeles | articles | horizon=6 | 38 splits complete
Tuned Merced | articles
Merced | articles | horizon=1 | 40 splits complete


In [94]:
XGB_predictions_df = pd.concat(
    [
        df.assign(
            County=county,
            Feature_Set=feature_name,
            Horizon=horizon
        )
        for (county, feature_name, horizon), df
        in XGB_results.items()
    ],
    ignore_index=True
)

XGB_predictions_df.to_csv(
    "xgboost_2001_predictions.csv",
    index=False
)

pd.DataFrame(XGB_importance_rows).to_csv(
    "xgboost_feature_importance_2001.csv",
    index=False
)

pd.DataFrame([
    {
        "County": county,
        "Feature_Set": feature_name,
        **params
    }
    for (county, feature_name), params in XGB_best_params.items()
]).to_csv(
    "xgboost_best_params_2001.csv",
    index=False
)

In [172]:
evaluation_rows = []

for (county, feature_name, horizon), df in XGB_results.items():
    actual = df["actual"].to_numpy()
    predicted = df["predicted"].to_numpy()

    evaluation_rows.append({
        "County": county,
        "Model": "XGBoost",
        "Feature Set": feature_name,
        "Horizon": horizon,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": (
            r2_score(actual, predicted)
            if len(actual) >= 2 else np.nan
        ),
        "N Predictions": len(actual)
    })

XBG_evaluations_table = pd.DataFrame(evaluation_rows)

XBG_evaluations_table = XBG_evaluations_table.sort_values(
    ["County", "Horizon", "Feature Set"]
).reset_index(drop=True)

XBG_evaluations_table.to_csv('xgboost_2001_Media_Evals.csv', index=False)

In [152]:
def select_sarimax_order(
        y_train,
        X_train,
        seasonal_period=12,
        trace=False,
):
    automatic_model = auto_arima(
        y=y_train,
        X=X_train,
        seasonal=True,
        m=seasonal_period,
        test="kpss",
        seasonal_test="ocsb",
        start_p=0,
        start_q=0,
        max_p=4,
        max_q=4,
        start_P=0,
        start_Q=0,
        max_P=2,
        max_Q=2,
        max_d=2,
        max_D=1,
        stepwise=True,
        information_criterion="aic",
        trace=trace,
    )

    trend = "c" if automatic_model.with_intercept else "n"

    return {
        "order": automatic_model.order,
        "seasonal_order": automatic_model.seasonal_order,
        "trend": trend
    }

In [159]:
def sarimax(X_train, Y_train, X_test, Y_test, params):

    model = SARIMAX(
        endog=Y_train,
        exog=X_train,
        order=params["order"],
        seasonal_order=params["seasonal_order"],
        trend=params["trend"]
    ).fit(disp=False)

    forecast = model.get_forecast(
        steps=len(Y_test),
        exog=X_test
    )

    return {
        "model": model,
        "forecast": forecast.predicted_mean,
        "order": params["order"],
        "seasonal_order": params["seasonal_order"],
        "trend": params["trend"],
        "converged": model.mle_retvals.get("converged", None)
    }

In [112]:
county = 'Alameda'

original = county_data[county].copy()
model = county_model_data[county]['df'].copy()

print("Original rows:", len(original))
print("Model rows:", len(model))

print("\nDates missing from model:")
print(set(original['year_month']) - set(model['year_month']))

print("\nDates missing from original:")
print(set(model['year_month']) - set(original['year_month']))

Original rows: 164
Model rows: 163

Dates missing from model:
{Period('2001-12', 'M')}

Dates missing from original:
set()


In [165]:
import warnings
warnings.filterwarnings('ignore')

In [160]:
SARIMAX_results = {}
SARIMAX_best_params = {}
SARIMAX_importance_rows = []

for county, feature_sets in county_splits.items():

    for feature_name, horizons_for_feature in feature_sets.items():

        for horizon, splits in horizons_for_feature.items():

            if not splits:
                print(f"Skipping {county} / {feature_name} / {horizon}")
                continue

            # Tune ONCE using the first training window
            first_split = splits[0]

            params = select_sarimax_order(
                first_split["y_train"],
                first_split["X_train"],
                seasonal_period=12,
                trace=False
            )

            SARIMAX_best_params[(county, feature_name, horizon)] = params

            print(
                f"Tuned {county} | {feature_name} | "
                f"horizon={horizon} | order={params['order']}"
            )

            forecasts = []

            # Reuse the selected parameters for every walk-forward split
            for i, split in enumerate(splits):

                result = sarimax(
                    X_train=split["X_train"],
                    Y_train=split["y_train"],
                    X_test=split["X_test"],
                    Y_test=split["y_test"],
                    params=params
                )

                predictions = result["forecast"]
                actuals = split["y_test"].to_numpy().ravel()

                for lead, (actual, predicted) in enumerate(
                    zip(actuals, predictions), start=1
                ):
                    forecasts.append({
                        "split": i,
                        "lead": lead,
                        "actual": actual,
                        "predicted": predicted,
                        "converged": result["converged"]
                    })

                # Store coefficients for each fitted split
                for feature, coefficient in zip(
                    split["X_train"].columns,
                    result["model"].params[
                        split["X_train"].columns
                    ]
                ):
                    SARIMAX_importance_rows.append({
                        "County": county,
                        "Feature_Set": feature_name,
                        "Horizon": horizon,
                        "Split": i,
                        "Feature": feature,
                        "Coefficient": coefficient
                    })

            SARIMAX_results[(county, feature_name, horizon)] = (
                pd.DataFrame(forecasts)
            )

            print(
                f"Completed {county} | {feature_name} | "
                f"horizon={horizon} | {len(splits)} splits"
            )


Tuned Alameda | articles | horizon=1 | order=(0, 0, 3)
Completed Alameda | articles | horizon=1 | 43 splits
Tuned Alameda | articles | horizon=3 | order=(0, 0, 3)
Completed Alameda | articles | horizon=3 | 41 splits
Tuned Alameda | articles | horizon=6 | order=(0, 0, 3)
Completed Alameda | articles | horizon=6 | 38 splits
Tuned Contra Costa | articles | horizon=1 | order=(0, 0, 2)
Completed Contra Costa | articles | horizon=1 | 43 splits
Tuned Contra Costa | articles | horizon=3 | order=(0, 0, 2)
Completed Contra Costa | articles | horizon=3 | 41 splits
Tuned Contra Costa | articles | horizon=6 | order=(0, 0, 2)
Completed Contra Costa | articles | horizon=6 | 38 splits
Tuned Fresno | articles | horizon=1 | order=(1, 0, 1)
Completed Fresno | articles | horizon=1 | 43 splits
Tuned Fresno | articles | horizon=3 | order=(1, 0, 1)
Completed Fresno | articles | horizon=3 | 41 splits
Tuned Fresno | articles | horizon=6 | order=(1, 0, 1)
Completed Fresno | articles | horizon=6 | 38 splits
Tune

In [167]:
smax_predictions_df = pd.concat(
    [
        df.assign(
            County=county,
            Feature_Set=feature_name,
            Horizon=horizon
        )
        for (county, feature_name, horizon), df
        in XGB_results.items()
    ],
    ignore_index=True
)

smax_predictions_df.to_csv(
    "SARIMAX_2001_predictions.csv",
    index=False
)

pd.DataFrame(SARIMAX_importance_rows).to_csv(
    "SARIMAX_feature_importance_2001.csv",
    index=False
)

pd.DataFrame([
    {
        "County": county,
        "Feature_Set": feature_name,
        **params
    }
    for (county, feature_name), params in SARIMAX_best_params.items()
]).to_csv(
    "SARIMAX_best_params_2001.csv",
    index=False
)

ValueError: too many values to unpack (expected 2, got 3)

In [168]:
evaluation_rows = []

for (county, feature_name, horizon), df in SARIMAX_results.items():
    actual = df["actual"].to_numpy()
    predicted = df["predicted"].to_numpy()

    evaluation_rows.append({
        "County": county,
        "Model": "SARIMAX",
        "Feature Set": feature_name,
        "Horizon": horizon,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": (
            r2_score(actual, predicted)
            if len(actual) >= 2 else np.nan
        ),
        "N Predictions": len(actual)
    })

SARIMAX_evaluations_table = pd.DataFrame(evaluation_rows)

SARIMAX_evaluations_table = SARIMAX_evaluations_table.sort_values(
    ["County", "Horizon", "Feature Set"]
).reset_index(drop=True)

SARIMAX_evaluations_table.to_csv('SARIMAX_2001_Media_Evals.csv', index=False)

In [161]:
def RandForest(Xtrain, Ytrain, Xtest, Ytest):

    # Make sure the target is one-dimensional
    Ytrain = np.asarray(Ytrain).reshape(-1)
    Ytest = np.asarray(Ytest).reshape(-1)

    fr = RandomForestRegressor(random_state=5)

    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 1.0]
    }

    tscv = TimeSeriesSplit(n_splits=5)

    grid_search = GridSearchCV(
        estimator=fr,
        param_grid=param_grid,
        cv=tscv,
        n_jobs=-1,
        scoring='neg_mean_squared_error',
        verbose=2
    )

    # Tune on training data only
    grid_search.fit(Xtrain, Ytrain)

    # Best estimator has already been fitted on Xtrain, Ytrain
    model = grid_search.best_estimator_

    # Forecast the held-out period
    fpreds = model.predict(Xtest)

    # Evaluate only against the corresponding test observations
    evaluation = Evaluator(Ytest, fpreds)

    return {
        'model': model,
        'evaluation': evaluation,
        'forecast': fpreds,
        'parameters': grid_search.best_params_,
        'cv_rmse': np.sqrt(-grid_search.best_score_)
    }

In [ ]:
forest_results = {}
forest_best_params = {}
forest_importance_rows = []

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 1.0]
}

for county, feature_sets_for_county in county_splits.items():

    for feature_name, horizons_for_feature in feature_sets_for_county.items():

        # Get the first 120-month training window for this county and feature set
        initial_splits = horizons_for_feature[1]

        if not initial_splits:
            print(f"Skipping {county} / {feature_name}: insufficient data")
            continue

        first_split = initial_splits[0]
        X_initial = first_split["X_train"]
        y_initial = first_split["y_train"].to_numpy().ravel()

        # Tune hyperparameters once per county and feature set
        search = GridSearchCV(
            RandomForestRegressor(random_state=5, n_jobs=1),
            param_grid=param_grid,
            cv=TimeSeriesSplit(n_splits=5),
            scoring="neg_mean_squared_error",
            n_jobs=-1,
            verbose=0
        )

        search.fit(X_initial, y_initial)
        best_params = search.best_params_

        forest_best_params[(county, feature_name)] = best_params

        # Evaluate each horizon using walk-forward splits
        for horizon, splits_for_horizon in horizons_for_feature.items():

            forecasts = []

            for i, split in enumerate(splits_for_horizon):

                X_train = split["X_train"]
                X_test = split["X_test"]
                y_train = split["y_train"]
                y_test = split["y_test"]

                model = RandomForestRegressor(
                    **best_params,
                    random_state=5,
                    n_jobs=-1
                )

                model.fit(X_train, y_train)
                predictions = model.predict(X_test)

                for actual, predicted in zip(
                    y_test.to_numpy().ravel(), predictions
                ):
                    forecasts.append({
                        "split": i,
                        "actual": actual,
                        "predicted": predicted
                    })

                for feature, importance in zip(
                    X_train.columns,
                    model.feature_importances_
                ):
                    forest_importance_rows.append({
                        "County": county,
                        "Feature_Set": feature_name,
                        "Horizon": horizon,
                        "Split": i,
                        "Feature": feature,
                        "Importance": importance
                    })

            # Store all predictions for this county/feature set/horizon
            forest_results[(county, feature_name, horizon)] = pd.DataFrame(
                forecasts
            )

            print(
                f"{county} | {feature_name} | horizon={horizon} | "
                f"{len(splits_for_horizon)} splits complete"
            )


Alameda | articles | horizon=1 | 43 splits complete
Alameda | articles | horizon=3 | 41 splits complete
Alameda | articles | horizon=6 | 38 splits complete
Contra Costa | articles | horizon=1 | 43 splits complete


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\z

Contra Costa | articles | horizon=3 | 41 splits complete
Contra Costa | articles | horizon=6 | 38 splits complete
Fresno | articles | horizon=1 | 43 splits complete
Fresno | articles | horizon=3 | 41 splits complete
Fresno | articles | horizon=6 | 38 splits complete
Kern | articles | horizon=1 | 43 splits complete
Kern | articles | horizon=3 | 41 splits complete
Kern | articles | horizon=6 | 38 splits complete


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\z

Los Angeles | articles | horizon=1 | 43 splits complete
Los Angeles | articles | horizon=3 | 41 splits complete
Los Angeles | articles | horizon=6 | 38 splits complete
Merced | articles | horizon=1 | 40 splits complete
Merced | articles | horizon=3 | 38 splits complete
Merced | articles | horizon=6 | 35 splits complete


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(


... (11 outputs left out)

c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\z

Tulare | articles | horizon=6 | 38 splits complete
Ventura | articles | horizon=1 | 43 splits complete


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\z

Ventura | articles | horizon=3 | 41 splits complete
Ventura | articles | horizon=6 | 38 splits complete


In [169]:
forest_predictions_df = pd.concat(
    [
        df.assign(
            County=county,
            Feature_Set=feature_name,
            Horizon=horizon
        )
        for (county, feature_name, horizon), df
        in forest_results.items()
    ],
    ignore_index=True
)

forest_predictions_df.to_csv(
    "RandFor_2001_predictions.csv",
    index=False
)

pd.DataFrame(forest_importance_rows).to_csv(
    "RandFor_feature_importance_2001.csv",
    index=False
)

pd.DataFrame([
    {
        "County": county,
        "Feature_Set": feature_name,
        **params
    }
    for (county, feature_name), params in forest_best_params.items()
]).to_csv(
    "RandFor_best_params_2001.csv",
    index=False
)

In [171]:
evaluation_rows = []

for (county, feature_name, horizon), df in forest_results.items():
    actual = df["actual"].to_numpy()
    predicted = df["predicted"].to_numpy()

    evaluation_rows.append({
        "County": county,
        "Model": "RandFor",
        "Feature Set": feature_name,
        "Horizon": horizon,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": (
            r2_score(actual, predicted)
            if len(actual) >= 2 else np.nan
        ),
        "N Predictions": len(actual)
    })

XBG_evaluations_table = pd.DataFrame(evaluation_rows)

XBG_evaluations_table = XBG_evaluations_table.sort_values(
    ["County", "Horizon", "Feature Set"]
).reset_index(drop=True)

XBG_evaluations_table.to_csv('Forest_2001_Media_Evals.csv', index=False)